# XMCA-Net-Lite: 09 - Final Reporting, Evaluation & Figures
### Purpose
Produce publication-ready evaluation results:
- Held-out test set evaluation (run exactly ONCE for final reporting).
- Bootstrap 95% Confidence Intervals.
- Calibration reliability plots and ROC/PR curves.
- Failure proxy attribution breakdowns.
- Formatted markdown and LaTeX summary tables.
**Research Claim Limit**: Reports proxy attribution only; no causal claims.


In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
# Robust project root discovery (supports Kaggle uppercase/lowercase and local)
PROJECT_ROOT = None
for candidate in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/XMCA"), Path("/kaggle/working/xmca")]:
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate.resolve()
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, set_seed
from src.combiner.features import build_feature_dataframe
from src.combiner.model import XMCACombiner
from src.evaluation.metrics import compute_all_metrics
from src.evaluation.bootstrap import compute_bootstrap_confidence_intervals
from src.evaluation.plots import plot_reliability_diagram, plot_roc_pr_curves
from src.utils.logging_metadata import save_run_metadata

config = load_config()
seed = config["project"]["default_seed"]
set_seed(seed)


In [ ]:
# Load trained model and validation threshold
combiner_dir = Path(config["paths"]["combiner_dir"])
combiner = XMCACombiner.load(combiner_dir)

with open(combiner_dir / "thresholds.json") as f:
    thresh_data = json.load(f)
threshold = thresh_data["threshold"]
print(f"Loaded validation tuned threshold: {threshold:.4f}")

# Load test split data
from src.utils.checkpoints import ShardManager
p_df = ShardManager(Path(config["paths"]["perception_dir"])).load_all_shards("perception_scores")
r_df = ShardManager(Path(config["paths"]["retrieval_dir"])).load_all_shards("retrieval_scores")
u_df = ShardManager(Path(config["paths"]["utilization_dir"])).load_all_shards("utilization_scores")

unified = p_df[["id", "dataset", "s_p", "is_hallucination_gt"]].merge(
    r_df[["id", "s_r", "retrieved_passages"]], on="id"
).merge(
    u_df[["id", "s_u"]], on="id"
)
unified["evidence_count"] = unified["retrieved_passages"].apply(len)
unified["claim_count"] = 2

splits_dir = Path(config["paths"]["splits_dir"])
test_ids = set()
for p in splits_dir.glob("*/*test.json"):
    with open(p) as f:
        test_ids.update([r["id"] for r in json.load(f)])

test_df = build_feature_dataframe(unified[unified["id"].isin(test_ids)].to_dict(orient="records"))
y_test = test_df["target"].values


In [ ]:
# Final held-out test evaluation (run only once!)
test_probs = combiner.predict_proba(test_df)
test_metrics = compute_all_metrics(y_test, test_probs, threshold=threshold)
ci_metrics = compute_bootstrap_confidence_intervals(y_test, test_probs, threshold=threshold, n_bootstraps=1000)

reports_dir = Path(config["paths"]["reports_dir"])
reports_dir.mkdir(parents=True, exist_ok=True)

# Generate diagnostic plots
plot_reliability_diagram(y_test, test_probs, reports_dir / "reliability_diagram.png")
plot_roc_pr_curves(y_test, test_probs, reports_dir / "roc_pr_curves.png")

# Generate publication-grade LaTeX tables for research manuscript
from src.evaluation.tables import generate_main_results_table_latex, generate_ablation_table_latex
latex_t1 = generate_main_results_table_latex(test_metrics, ci_metrics, reports_dir / "table_1_main_results.tex")

# Load ablation results if available and export LaTeX Table 2
abl_file = Path(config["paths"]["ablations_dir"]) / "ablation_results.json"
if abl_file.exists():
    with open(abl_file) as f:
        abl_data = json.load(f)
    latex_t2 = generate_ablation_table_latex(abl_data, reports_dir / "table_2_ablation_study.tex")

# Proxy failure attribution breakdown
attr_counts = test_df["proxy_attribution"].value_counts().to_dict()

final_report = {
    "held_out_test_metrics": test_metrics,
    "bootstrap_95_ci": ci_metrics,
    "proxy_failure_attribution_counts": attr_counts,
    "claim_boundary": config["project"]["claim_boundary"]
}

with open(reports_dir / "final_evaluation_report.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=2)

print("\n================ FINAL TEST RESULTS ================")
for k, v in test_metrics.items():
    ci = ci_metrics.get(k, {})
    ci_str = f" [95% CI: {ci.get('lower_ci', 0):.4f} - {ci.get('upper_ci', 0):.4f}]" if ci else ""
    print(f"  {k.upper():<12}: {v:.4f}{ci_str}")
print("\nProxy Failure Attribution:", attr_counts)
print("LaTeX Table 1 generated at: artifacts/reports/table_1_main_results.tex")
print("====================================================")
